# Simulation Validation: Comissão vs Desconto Otimizado

Este notebook tem como objetivo validar a simulação completa de negociações comissionadas para os expositores que já foram fechados com modelo de comissão no evento RJ_26.

A ideia central é responder três perguntas práticas para cada expositor comissionado:
1. Quanto ele precisa vender pós-evento para a empresa ter a mesma (ou melhor) receita que teria com um desconto otimizado?
2. Qual é a previsão realista de vendas pós-evento segundo o modelo de Trends?
3. Em quantos cenários (pessimista / base / otimista) ele consegue bater o break-even?

Com isso, conseguimos identificar quais comissões estão bem estruturadas e quais estão gerando risco de perda de receita.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()

DB_URI = os.getenv("DB_URI")
engine = create_engine(DB_URI)

query_comissionados = """
SELECT 
  nome_fantasia,
  area,
  receita_realizada,
  percentual_comissao,
  (receita_realizada / percentual_comissao) + receita_realizada AS valor_excedente
FROM expositores_atual 
WHERE pipeline = 'RJ_26' 
  AND percentual_comissao > 0
"""

df = pd.read_sql(query_comissionados, engine)

print(f"Total de expositores comissionados: {len(df)}")
df.head()

Total de expositores comissionados: 11


,nome_fantasia,area,receita_realizada,percentual_comissao,valor_excedente
0,AKKO,330.0,30000.0,0.05,630000.0
1,ALIKKA MAKEUP,20.0,10000.0,0.10,110000.0
2,APPROVE,120.0,30000.0,0.10,330000.0
3,LEHUA,60.0,15000.0,0.10,165000.0
4,LICOR D'BELÉM,20.0,10000.0,0.10,110000.0


Vamos agora, simular descontos por faixas uniformementes contínuas com base nos quartis obtidos em outros estudos por área.

In [2]:
def definir_porte(area):
    if area <= 20:
        return "Pequeno"
    elif area <= 35:
        return "Médio"
    else:
        return "Grande"

df['porte'] = df['area'].apply(definir_porte)

np.random.seed(27)

def simular_desconto(porte):
    if porte == "Pequeno":
        return np.random.uniform(0.05, 0.20)
    elif porte == "Médio":
        return np.random.uniform(0.20, 0.25)
    else:  # Grande
        return np.random.uniform(0.25, 0.70)

df['desconto_simulado'] = df['porte'].apply(simular_desconto)

preco_base = 690.0
df['receita_otimizada_nova'] = (df['area'] * preco_base) * (1 - df['desconto_simulado'])

df['diferenca_receita'] = df['receita_otimizada_nova'] - df['receita_realizada']
df['ganho_percentual'] = (df['diferenca_receita'] / df['receita_realizada']) * 100

resumo = df.groupby('porte').agg({
    'area': 'mean',
    'desconto_simulado': 'mean',
    'receita_realizada': 'sum',
    'receita_otimizada_nova': 'sum',
    'diferenca_receita': 'sum',
    'ganho_percentual': 'mean'
}).round(2)

print(resumo)
print(f"\nGanho total estimado na receita: R$ {df['diferenca_receita'].sum():,.2f}")
print(f"Impacto percentual médio: {df['ganho_percentual'].mean():.1f}%")

           area  desconto_simulado  receita_realizada  receita_otimizada_nova  \
porte                                                                           
Grande   103.12               0.57           153750.0               269631.56   
Médio     30.00               0.23            15000.0                15873.65   
Pequeno   20.00               0.14            20000.0                23740.21   

         diferenca_receita  ganho_percentual  
porte                                         
Grande           115881.56             51.60  
Médio               873.65              5.82  
Pequeno            3740.21             18.70  

Ganho total estimado na receita: R$ 120,495.42
Impacto percentual médio: 41.5%


A simualão realizada novamente, comprova mais uma vez que, otimizar descontos controlados por faixas também controladas, mesmo que aleatórias, resultam em uma melhora significativa na receita, mesmo que para apenas clientes comissionados.

Agora vamos coletar os dados de trends do Google com pytrends e rodar nossos modelos.

In [3]:
from pytrends.request import TrendReq
import time
pytrends = TrendReq(hl='pt-BR', tz=360)
kw_list_all = df['nome_fantasia'].unique().tolist() 

all_data_frames = []

for expositor in kw_list_all:
    try:
        print(f"Processando: {expositor}")
        
        pytrends.build_payload(
            [expositor], 
            cat=0, 
            timeframe='today 5-y', 
            geo='BR-RJ'
        )
        
        data = pytrends.interest_over_time()
        
        if not data.empty:
            if 'isPartial' in data.columns:
                data = data.drop(columns=['isPartial'])
            
            data = data.reset_index()
            
            data = data.rename(columns={
                'date': 'data',
                expositor: 'trends'
            })
            
            data['nome_expositor'] = expositor
            
            all_data_frames.append(data[['data', 'nome_expositor', 'trends']])
        
        time.sleep(3)
    
    except Exception as e:
        print(f"Erro em {expositor}: {e}")

if all_data_frames:
    df_trends = pd.concat(all_data_frames, ignore_index=True)
    
    df_trends = df_trends.groupby('nome_expositor').filter(
        lambda x: x['trends'].sum() > 0
    )
    

stats = df_trends.groupby('nome_expositor')['trends'].agg(
    pct_zero=(lambda x: (x == 0).mean() * 100)
).reset_index()

expositores_otimos = stats[stats['pct_zero'] < 70]["nome_expositor"]
expositores_intermitentes = stats[(stats['pct_zero'] < 95) & (stats['pct_zero'] >= 70)]['nome_expositor']

df_otimo = df_trends[df_trends['nome_expositor'].isin(expositores_otimos)]
df_intermitente = df_trends[df_trends['nome_expositor'].isin(expositores_intermitentes)]

df_model_otimo = df_otimo.rename(columns={
    'data': 'ds',
    'trends': 'y',
    'nome_expositor': 'ID'
})
df_model_otimo["y"] = np.log1p(df_model_otimo["y"])

df_model_intermitente = df_intermitente.rename(columns={
    'data': 'ds',
    'trends': 'y',
    'nome_expositor': 'ID'
})

df_model_intermitente["y"] = np.log1p(df_model_intermitente["y"])

def croston(series, alpha=0.2):
    """
    Implementação clássica do método de Croston.
    """
    d = series[series > 0]
    if len(d) == 0:
        return 0.0
    
    indices = np.where(series > 0)[0]
    p = np.diff(indices)

    a = d[0]
    q = p[0] if len(p) > 0 else 1.0
    
    for i in range(1, len(d)):
        a = alpha * d[i] + (1 - alpha) * a
        if i - 1 < len(p):
            q = alpha * p[i-1] + (1 - alpha) * q
            
    return a / q


def predict_intermitente(df, id_col, target_col, date_col, predict_date, alpha=0.2):
    rows = []
    
    for id_ in df[id_col].unique():
        df_id = df[df[id_col] == id_].sort_values(date_col)
        history_df = df_id[df_id[date_col] < predict_date]
        
        if len(history_df) > 0:
            history_values = history_df[target_col].values
            
            yhat = croston(history_values, alpha=alpha)
            
            rows.append({
                id_col: id_,
                "ds": predict_date,
                "yhat": yhat,
                "model": "CROSTON_0.2"
            })
            
    return pd.DataFrame(rows)

from neuralprophet import load
import torch
import neuralprophet
import functools

torch.load = functools.partial(torch.load, weights_only=False)

model_otimo = load("../models/model_otimo.np")

Processando: AKKO
Processando: ALIKKA MAKEUP
Processando: APPROVE
Processando: LEHUA
Processando: LICOR D'BELÉM
Processando: CCM
Processando: FIRST CLASS HOME
Processando: FLOREST
Processando: RYGY
Processando: USE MIRRA
Processando: VIA MIA


Importing plotly failed. Interactive plots will not work.
Importing plotly failed. Interactive plots will not work.


In [43]:
from neuralprophet import load
import torch
import neuralprophet
import functools

def croston(series, alpha=0.2):
    """
    Implementação clássica do método de Croston.
    """
    d = series[series > 0]
    if len(d) == 0:
        return 0.0
    
    indices = np.where(series > 0)[0]
    p = np.diff(indices)

    a = d[0]
    q = p[0] if len(p) > 0 else 1.0
    
    for i in range(1, len(d)):
        a = alpha * d[i] + (1 - alpha) * a
        if i - 1 < len(p):
            q = alpha * p[i-1] + (1 - alpha) * q
            
    return a / q


def predict_intermitente(df, id_col, target_col, date_col, predict_date, alpha=0.2):
    rows = []
    
    for id_ in df[id_col].unique():
        df_id = df[df[id_col] == id_].sort_values(date_col)
        history_df = df_id[df_id[date_col] < predict_date]
        
        if len(history_df) > 0:
            history_values = history_df[target_col].values
            
            yhat = croston(history_values, alpha=alpha)
            
            rows.append({
                id_col: id_,
                "ds": predict_date,
                "yhat": yhat,
                "model": "CROSTON_0.2"
            })
            
    return pd.DataFrame(rows)

torch.load = functools.partial(torch.load, weights_only=False)

model_otimo = load("../models/model_otimo.np")

def predict_hibrido(df_otimo, df_intermitente, model_neural, predict_date):
    """
    Executa predições usando NeuralProphet para séries contínuas 
    e Croston para séries intermitentes.
    """
    results = []
    data_alvo = pd.to_datetime(predict_date)

   # --- 1. MODELO ÓTIMO ---

    if not df_otimo.empty:
        print(f"Predizendo {df_otimo['ID'].nunique()} expositores contínuos...")
        
        ultima_data_treino = pd.to_datetime(df_otimo['ds']).max()
        semanas_a_frente = (data_alvo - ultima_data_treino).days // 7
        
        if semanas_a_frente <= 0:
            forecast_otimo = model_neural.predict(df_otimo)
            col_target = 'yhat1' # No passado/presente usamos a primeira
        else:
            # Cria o futuro
            future = model_neural.make_future_dataframe(df_otimo, periods=semanas_a_frente)
            forecast_otimo = model_neural.predict(future)
            col_target = f'yhat{semanas_a_frente}'
            
        # Filtra a data e verifica se a coluna existe
        pred_otimo = forecast_otimo[forecast_otimo['ds'] == predict_date].copy()
        
        if not pred_otimo.empty and col_target in pred_otimo.columns:
            pred_otimo['yhat'] = np.expm1(pred_otimo[col_target])
            pred_otimo['model'] = 'NEURAL_PROPHET_OTIMO'
            results.append(pred_otimo[['ID', 'ds', 'yhat', 'model']])
        else:
            print(f"Aviso: Coluna {col_target} não encontrada para a data {predict_date}")

    # --- 2. MODELO INTERMITENTE ---
    if not df_intermitente.empty:
        print(f"Predizendo {df_intermitente['ID'].nunique()} expositores intermitentes...")
        pred_croston = predict_intermitente(df_intermitente, 'ID', 'y', 'ds', predict_date)
        pred_croston['yhat'] = np.expm1(pred_croston['yhat'])
        results.append(pred_croston)

    # --- 3. CONSOLIDAÇÃO ---

    if results:
        df_final = pd.concat(results, ignore_index=True)
        df_final['yhat'] = df_final['yhat'].clip(lower=0)
        return df_final
    else:
        return pd.DataFrame()
    

def preparar_para_predict(df, lags):
    df['ds'] = pd.to_datetime(df['ds'])
    
    df = df.sort_values(['ID', 'ds']).drop_duplicates()
    
    df_lista = []
    for id_ in df['ID'].unique():
        temp = df[df['ID'] == id_].set_index('ds')
        temp = temp.asfreq('W-SUN') 
        temp['y'] = temp['y'].fillna(method='ffill')
        temp['ID'] = id_
        df_lista.append(temp.reset_index())
    
    return pd.concat(df_lista)

df_model_otimo = preparar_para_predict(df_model_otimo, 14)

In [48]:
df_previsoes = predict_hibrido(df_model_otimo, df_model_intermitente, model_otimo, '2026-09-20')

df_previsoes['nome_fantasia'] = df_previsoes["ID"]

df_previsoes

INFO - (NP.df_utils._infer_frequency) - Major frequency W-SUN corresponds to 99.618% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - W


WARNING - (NP.data.splitting._make_future_dataframe) - Number of forecast steps is defined by n_forecasts. Adjusted to 26.
INFO - (NP.df_utils._infer_frequency) - Major frequency W-SUN corresponds to 99.618% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - W
WARNING - (NP.data.splitting._make_future_dataframe) - Number of forecast steps is defined by n_forecasts. Adjusted to 26.
INFO - (NP.df_utils._infer_frequency) - Major frequency W-SUN corresponds to 99.618% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - W
WARNING - (NP.data.splitting._make_future_dataframe) - Number of forecast steps is defined by n_forecasts. Adjusted to 26.
INFO - (NP.df_utils._infer_frequency) - Major frequency W-SUN corresponds to 97.619% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - W
INFO - (NP.df_utils._infer_frequency) - Major frequency W-SUN corre

Predizendo 3 expositores contínuos...
Predicting DataLoader 0: 100%|██████████| 1/1 [00:00<00:00, 453.10it/s]
Predizendo 2 expositores intermitentes...


,ID,ds,yhat,model,nome_fantasia
0,APPROVE,2026-09-20 00:00:00,33.431255,NEURAL_PROPHET_OTIMO,APPROVE
1,CCM,2026-09-20 00:00:00,45.032715,NEURAL_PROPHET_OTIMO,CCM
2,VIA MIA,2026-09-20 00:00:00,15.704706,NEURAL_PROPHET_OTIMO,VIA MIA
3,AKKO,2026-09-20,39.616735,CROSTON_0.2,AKKO
4,FLOREST,2026-09-20,2.659097,CROSTON_0.2,FLOREST


In [71]:
df_final = pd.merge(df, df_previsoes, on="nome_fantasia", how='left').drop(columns=["ID", "ds"]).dropna(subset=['yhat'])

ticket_medio = {'AKKO': 250, 'APPROVE': 200, 'CCM': 220, 'FLOREST': 180, "VIA MIA": 150}

def simular_conversao_funil(porte):
    if porte == "Pequeno":
        return np.random.uniform(0.15, 0.25), np.random.uniform(0.05, 0.10)
    elif porte == "Médio":
        return np.random.uniform(0.10, 0.18), np.random.uniform(0.03, 0.07)
    return np.random.uniform(0.05, 0.12), np.random.uniform(0.01, 0.04)

df_final['ticket_medio'] = df_final['nome_fantasia'].map(ticket_medio)
df_final[['taxa_visita', 'taxa_venda']] = pd.DataFrame(df_final['porte'].apply(simular_conversao_funil).tolist(), index=df_final.index)

df_final['vendas'] = df_final['yhat'] * 100000 * df_final['taxa_visita'] * df_final['taxa_venda']
df_final['receita_estimada'] = df_final['vendas'] * df_final['ticket_medio']
df_final['meta'] = df_final['receita_otimizada_nova'] + df_final['valor_excedente']
df_final['ganho_real'] = df_final['receita_estimada'] - df_final['valor_excedente']
df_final['resumo'] = np.where(df_final['receita_estimada'] > df_final['meta'], "Comissão Vale A Pena", "Comissão Não Vale A Pena")

df_pronto = df_final[['nome_fantasia', 'receita_estimada', 'meta', 'ganho_real', 'receita_realizada', 'resumo']]

df2 = df[~df['nome_fantasia'].isin(df_pronto['nome_fantasia'])].copy()

df2['meta'] = df2['receita_otimizada_nova'] + df2['valor_excedente']
df2['receita_estimada'] = 0  # Não há previsão
df2['ganho_real'] = -df2['valor_excedente'] # O "ganho" é o prejuízo do excedente não batido
df2['resumo'] = "Risco Alto: Dados Insuficientes (Fixo Recomendado)"

df2_final = df2[['nome_fantasia', 'receita_estimada', 'meta', 'ganho_real', 'receita_realizada', 'resumo']]

df_entrega = pd.concat([df_pronto, df2_final], ignore_index=True)

pd.options.display.float_format = 'R$ {:,.2f}'.format

df_entrega.sort_values(by='ganho_real', ascending=False)


,nome_fantasia,receita_estimada,meta,ganho_real,receita_realizada,resumo
2,CCM,"R$ 3,827,313.49","R$ 177,802.72","R$ 3,662,313.49","R$ 15,000.00",Comissão Vale A Pena
0,AKKO,"R$ 2,453,486.63","R$ 757,153.46","R$ 1,823,486.63","R$ 30,000.00",Comissão Vale A Pena
1,APPROVE,"R$ 1,951,676.52","R$ 364,699.10","R$ 1,621,676.52","R$ 30,000.00",Comissão Vale A Pena
4,VIA MIA,"R$ 461,919.56","R$ 179,528.89","R$ 296,919.56","R$ 15,000.00",Comissão Vale A Pena
5,ALIKKA MAKEUP,R$ 0.00,"R$ 121,423.81","R$ -110,000.00","R$ 10,000.00",Risco Alto: Dados Insuficientes (Fixo Recomend...
7,LICOR D'BELÉM,R$ 0.00,"R$ 122,316.40","R$ -110,000.00","R$ 10,000.00",Risco Alto: Dados Insuficientes (Fixo Recomend...
3,FLOREST,"R$ 45,827.36","R$ 240,178.76","R$ -160,422.64","R$ 18,750.00",Comissão Não Vale A Pena
6,LEHUA,R$ 0.00,"R$ 179,879.10","R$ -165,000.00","R$ 15,000.00",Risco Alto: Dados Insuficientes (Fixo Recomend...
9,RYGY,R$ 0.00,"R$ 182,229.75","R$ -165,000.00","R$ 15,000.00",Risco Alto: Dados Insuficientes (Fixo Recomend...
10,USE MIRRA,R$ 0.00,"R$ 180,873.65","R$ -165,000.00","R$ 15,000.00",Risco Alto: Dados Insuficientes (Fixo Recomend...


## O Veredito dos Dados

As Locomotivas (CCM, AKKO, APPROVE): Essas marcas são o porto seguro do comissionamento. O potencial de ganho real acima de R$ 1,5 milhão justifica qualquer esforço de marketing cooperado.

A "Zona de Alerta" (FLOREST): O modelo foi cirúrgico aqui. O interesse é baixo demais para a meta estipulada. É o caso clássico de renegociação para fixo.

Os "Invisíveis Digitais" (ALIKKA, RYGY, etc.): O seu ganho_real negativo (travado no valor do excedente) é o escudo do financeiro. Sem dados, não há aposta no variável.

# Conclusão Estudo: Validação de Modelos e Simulação de Receita

O presente estudo validou a hipótese de que a otimização de descontos controlados por faixas resulta em uma melhora significativa na receita, mesmo para clientes comissionados. A utilização do modelo híbrido de séries temporais — integrando NeuralProphet para dados contínuos e Croston para dados intermitentes — permitiu gerar previsões realistas fundamentadas no interesse de busca do Google Trends. Os achados demonstram que expositores como CCM, AKKO e APPROVE apresentam alto potencial comercial, superando amplamente as metas de break-even projetadas. Em contrapartida, marcas com baixo volume de dados digitais foram classificadas como de "Risco Alto", recomendando-se a migração para modelos de aluguel fixo para garantir a segurança financeira do evento. Esta metodologia estruturada prova-se eficaz para apoiar decisões estratégicas de negociação sob incerteza, diferenciando claramente onde o modelo comissionado é vantajoso e onde ele representa um risco de perda de receita.